# DIMER Notebook: Vision-Language Retrieval and Cross-Modal Reranking

**Profile:** `MULTI-CAPABILITY`  
**Mode:** `WORKSHOP`  
**Notebook specification:** `2.1`  
**Status:** Candidate notebook carrier  
**Default tier:** `STANDARD`  
**Canonical runtime:** NVIDIA Tesla T4 or equivalent

This notebook demonstrates the retrieval architecture used by practical multimodal search systems:

```text
                    ┌─ SigLIP 2 ─┐
caption/image query ├─ SigLIP v1 ├─→ vector similarity → top-K
                    └─ BLIP ITC ─┘                    │
                                                      ↓
                                              BLIP ITM reranker
                                                      │
                                                      ↓
                                                final ranking
```

## Capability A — scalable dual-encoder retrieval

Compare frozen:

- **SigLIP 2 Base P16-224**
- **SigLIP v1 Base P16-256**
- **BLIP ITC**

on one shared real photograph-caption gallery.

## Capability B — cross-modal reranking

Use **BLIP ITM** to rescore only the top-K candidates from each first-stage retriever.

This tests the application-level question:

> How much richer pairwise reasoning is worth paying for after a cheap vector index has already narrowed the search space?

## Capability C — BLIP retrieval adaptation (`FULL`)

Fine-tune BLIP's last two fused text blocks, its ITC projections, and ITM head on the training image-caption pairs; select by validation `rsum`; then test both the adapted coarse retriever and adapted reranker.

SigLIP adaptation is deliberately excluded because the live DIMER SigLIP adapters are class-prompt adaptation contracts, not general multi-caption retrieval adapters.

### Standalone contract

The default path:

- uses pinned upstream Hugging Face snapshots directly;
- does not clone or fetch DIMER repository source;
- does not call DIMER workers or services;
- requires no token/login;
- downloads one digest-pinned VizWiz parquet shard and extracts only its first two row groups;
- implements retrieval evaluation, reranking, BLIP adaptation, artifacts, and BYOD locally in notebook cells.

All measurements are **sample-sanity evidence on one seeded corpus split**, not a benchmark claim.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to multimodal embeddings, retrieval, or reranking.

**Runtime.** Use the documented GPU runtime for the multi-model path.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`image/caption → dual-encoder embeddings → coarse retrieval → cross-modal reranking → ranked gallery`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.


## 0. Learning objectives

By the end of the notebook, participants should be able to:

1. distinguish retrieval from classification and caption generation;
2. explain dual-encoder versus cross-encoder/fused retrieval;
3. build an image and text embedding index;
4. evaluate image→text and text→image Recall@K;
5. interpret median rank and `rsum`;
6. explain why gallery size changes retrieval difficulty;
7. compute a reranker candidate oracle@K;
8. use BLIP ITM to rerank top-K candidates from a cheaper retriever;
9. inspect hard negatives and model disagreements;
10. compare embedding dimensions, storage, indexing cost, search latency, and reranking cost;
11. perform bounded BLIP retrieval adaptation without touching the vision encoder;
12. export/reload both embedding indexes and a BLIP adapter with provenance.

## 1. Notebook controls

In [ ]:
# @title Workshop controls
WORKSHOP_TIER = "STANDARD"  # @param ["STANDARD", "FULL"]

USE_BYOD = False            # @param {type:"boolean"}
BYOD_ZIP_PATH = ""          # @param {type:"string"}

OUTPUT_DIR = "outputs"      # @param {type:"string"}

RERANK_TOP_K = 5
RECALL_KS = (1, 5, 10)
GALLERY_SIZES = (70, 128, 256, 391)

BLIP_EPOCHS = 4
BLIP_LEARNING_RATE = 2e-5
BLIP_BATCH_SIZE = 16
BLIP_TRAINABLE_TEXT_LAYERS = 2
SEED = 0
SPLIT_SEED = 42

if WORKSHOP_TIER not in {"STANDARD", "FULL"}:
    raise ValueError("WORKSHOP_TIER must be STANDARD or FULL")
if RERANK_TOP_K not in (5, 10, 20):
    raise ValueError("RERANK_TOP_K must be one of 5, 10, 20")

from pathlib import Path
OUTPUT_ROOT = Path(OUTPUT_DIR)
WORK_ROOT = Path("work")

for p in [
    OUTPUT_ROOT / "data",
    OUTPUT_ROOT / "index",
    OUTPUT_ROOT / "validation",
    OUTPUT_ROOT / "frozen",
    OUTPUT_ROOT / "test",
    OUTPUT_ROOT / "adaptation" / "blip",
    OUTPUT_ROOT / "artifacts" / "blip",
    OUTPUT_ROOT / "figures",
    OUTPUT_ROOT / "provenance",
    WORK_ROOT / "models" / "siglip2",
    WORK_ROOT / "models" / "siglip1",
    WORK_ROOT / "models" / "blip",
    WORK_ROOT / "vizwiz" / "images",
    WORK_ROOT / "byod",
]:
    p.mkdir(parents=True, exist_ok=True)

print({
    "tier": WORKSHOP_TIER,
    "rerank_top_k": RERANK_TOP_K,
    "gallery_sizes": GALLERY_SIZES,
})

## 2. Install one pinned runtime

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Install runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    "torch==2.14.0",
    "torchvision==0.29.0",
    "torchaudio==2.11.0",
    "transformers==4.57.6",
    "huggingface-hub==0.36.2",
    "safetensors==0.8.0",
    "numpy==2.5.3",
    "pillow==11.3.0",
    "pyarrow==25.0.1",
    "matplotlib>=3.9,<3.11",
    "pandas>=2.2,<3.1",
]

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
if not SKIP_INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PINS], check=True)
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from PIL import Image, ImageOps
import io
import hashlib
import json
import random
import time
import gc

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": importlib.metadata.version("transformers"),
    "pyarrow": importlib.metadata.version("pyarrow"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

# 3. Immutable model identities

## SigLIP 2 Base P16-224

`google/siglip2-base-patch16-224`

Revision:

`5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2`

Primary weight:

- `model.safetensors`
- 1,500,800,904 bytes
- SHA-256 `612923381c76ec5a9bed335d1c48827e3f2e506ac31b044b63b2031fadee6a0b`

Parameters:

approximately **375.19M**

Embedding dimension:

**768**

## SigLIP v1 Base P16-256

`google/siglip-base-patch16-256`

Revision:

`b078df89e446d623010d890864d4207fe6399f61`

Primary weight:

- 812,856,640 bytes
- SHA-256 `f0cee7c815135c44a515eff72ab3040499744920442bc25567cd04efc93f8f65`

Parameters:

approximately **203.20M**

Embedding dimension:

**768**

## BLIP ITM Base COCO

`Salesforce/blip-itm-base-coco`

Revision:

`bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c`

Primary PyTorch weight:

- `pytorch_model.bin`
- 895,139,697 bytes
- SHA-256 `017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f`

Parameters:

approximately **223.74M**

ITC embedding dimension:

**256**

BLIP's base checkpoint is a pinned PyTorch pickle; this notebook verifies exact bytes and loads it through Transformers with `weights_only=True`.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Snapshot manifests
SIGLIP2 = {
    "model_id":"google/siglip2-base-patch16-224",
    "revision":"5ffaac51d5e2f3367f7dab0cad4be4cb07c0caa2",
    "files":[
        ("README.md",3367,"77fd3e4cc34abaa0cf891f03fc73be58cd0e36ce09741cb0fe096342e4d8867f"),
        ("config.json",253,"fe8b5fe6d5734360678fd71c11c21e1ea3364bd8598d34295d9206335973ffd7"),
        ("model.safetensors",1500800904,"612923381c76ec5a9bed335d1c48827e3f2e506ac31b044b63b2031fadee6a0b"),
        ("preprocessor_config.json",394,"9b36b57ebaf20f09bf4c22100ccc21877ea6bfe5aead0c00c59f8af8ccefacfc"),
        ("special_tokens_map.json",636,"baec30ea10906f16adb8c18af7a34023002c1746542612b8b41c9f09e1351351"),
        ("tokenizer.json",34363039,"cb9140fae3ac5122c972d37adf83e1248471a38147ad76f8215c8872c6fd8322"),
        ("tokenizer.model",4241003,"61a7b147390c64585d6c3543dd6fc636906c9af3865a5548f27f31aee1d4c8e2"),
        ("tokenizer_config.json",47164,"14afe629fe4959b9e0d51e1852b8d9f7ad074f90a1a7125a4fcdd17f06e78fc8"),
    ],
    "dim":768,
}
SIGLIP1 = {
    "model_id":"google/siglip-base-patch16-256",
    "revision":"b078df89e446d623010d890864d4207fe6399f61",
    "files":[
        ("README.md",4116,"b105a3cfa97df9f082c42817c3caf61bfd7d360386fed4bdebc0c7fff0acba01"),
        ("config.json",322,"acc261689fe8d29cf8ceacd5dc5d05fd53d2a43f3f6c57e6c16112a14a618dfa"),
        ("model.safetensors",812856640,"f0cee7c815135c44a515eff72ab3040499744920442bc25567cd04efc93f8f65"),
        ("preprocessor_config.json",368,"e12b577bd0da1f9bf6a0b8d3d78c8f9b55b9d99dec25556954e19cfc7b8ecf28"),
        ("special_tokens_map.json",409,"2b6a1ff67a27e0df9ac0c7d93250fc0d87431c7b366b3d5669217104f9088a26"),
        ("spiece.model",798330,"1e5036bed065526c3c212dfbe288752391797c4bb1a284aa18c9a0b23fcaf8ec"),
        ("tokenizer.json",2399357,"c6e405cb7c670d56636a9402c81023a55bc6c3c53d89cf02b92f5c5005bfe920"),
        ("tokenizer_config.json",711,"d6423dae508cc3a129d22ea443841c111832a1a73125b8f25ea8736951698bcb"),
    ],
    "dim":768,
}
BLIP = {
    "model_id":"Salesforce/blip-itm-base-coco",
    "revision":"bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c",
    "files":[
        ("README.md",5492,"db2e7ff1e647bc42d8b0d4cd3653ad65c1d24b5559a153e6e6ea2f3c12edd978"),
        ("config.json",4560,"3e6464c2ce7c54512ddb101c5e9a8e77f4c2d637be9e3d005667ccd4a34c6ef2"),
        ("preprocessor_config.json",445,"0aa66e2e9ac3ea3b5cd4388c35072e22db4e1cc1f96c7872bed07749c712ade1"),
        ("pytorch_model.bin",895139697,"017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f"),
        ("special_tokens_map.json",125,"b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"),
        ("tokenizer.json",711396,"d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"),
        ("tokenizer_config.json",456,"86da6fdb761b02f73a05561aba71711c2d7c205fe1fd1744046173a410263925"),
        ("vocab.txt",231508,"07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"),
    ],
    "dim":256,
}

# 4. Canonical corpus — VizWiz-Captions

Source:

`mm-eval/VizWiz-Captions`

Revision:

`c4a6d897836e7885d0095134f92d392e4e770539`

Pinned shard:

`data/val-00004-of-00005.parquet`

- 392,245,504 bytes
- SHA-256 `4492465a41d32b3c12b8b7b6a0cf7e0a0e202a5b825b006ca0c85dcdf24efd3e`

The notebook downloads that exact shard, verifies it as one immutable byte object, and then reads only row groups 0 and 1.

Expected captioned photographs:

- row group 0: **318**
- row group 1: **321**

Split:

- train: **208**
- validation: **40**
- core test: **70**
- gallery-only test additions: **321**
- full test gallery: **391 photographs / 1,737 captions**

Every photograph's captions remain in the same role.

In [ ]:
# @title Fetch and verify the full pinned parquet shard
from huggingface_hub import hf_hub_download

CORPUS_REPO="mm-eval/VizWiz-Captions"
CORPUS_REVISION="c4a6d897836e7885d0095134f92d392e4e770539"
CORPUS_PATH="data/val-00004-of-00005.parquet"
CORPUS_BYTES=392_245_504
CORPUS_SHA256="4492465a41d32b3c12b8b7b6a0cf7e0a0e202a5b825b006ca0c85dcdf24efd3e"

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):
            h.update(chunk)
    return h.hexdigest()

shard_path=Path(hf_hub_download(
    repo_id=CORPUS_REPO,
    repo_type="dataset",
    filename=CORPUS_PATH,
    revision=CORPUS_REVISION,
    local_dir=str(WORK_ROOT/"vizwiz"),
))
if shard_path.stat().st_size!=CORPUS_BYTES or sha256_file(shard_path)!=CORPUS_SHA256:
    raise ValueError("VizWiz parquet digest/size mismatch")

print({
    "path":str(shard_path),
    "bytes":shard_path.stat().st_size,
    "sha256":sha256_file(shard_path),
})

In [ ]:
# @title Decode row groups 0–1 and reproduce the 208/40/70 + 321 split
pf=pq.ParquetFile(shard_path)

def clean_captions(answer):
    out=[]
    for c in answer or []:
        text=" ".join(str(c).split())
        if text and len(text)<=256 and text not in out:
            out.append(text)
    return out

def decoded_rgb_digest(payload):
    with Image.open(io.BytesIO(payload)) as im:
        rgb=im.convert("RGB")
        h=hashlib.sha256()
        h.update(f"{rgb.width}x{rgb.height}".encode())
        h.update(rgb.tobytes())
        return h.hexdigest()

def read_group(group):
    table=pf.read_row_group(
        group,
        columns=["id","answer","question_type","text_detected","media"],
    )
    rows=[]
    for row in table.to_pylist():
        captions=clean_captions(row["answer"])
        media=row["media"]
        if not isinstance(media,list) or len(media)!=1:
            raise ValueError(f"{row['id']}: expected exactly one image")
        payload=bytes(media[0]["bytes"])
        image_id=str(row["id"])
        dest=WORK_ROOT/"vizwiz"/"images"/f"{image_id}.jpg"
        dest.write_bytes(payload)
        # Decode now so corrupt images fail before model acquisition.
        with Image.open(dest) as im:
            im.verify()
        rows.append({
            "image_id":image_id,
            "path":str(dest),
            "captions":captions,
            "category":"text" if bool(row["text_detected"]) else "no-text",
            "question_type":str(row["question_type"]),
            "file_sha256":hashlib.sha256(payload).hexdigest(),
            "pixel_sha256":decoded_rgb_digest(payload),
            "row_group":group,
        })
    return rows

group0=[r for r in read_group(0) if r["captions"]]
group1=[r for r in read_group(1) if r["captions"]]

if len(group0)!=318 or len(group1)!=321:
    raise RuntimeError(f"captioned-row counts drifted: {len(group0)} / {len(group1)}")

pool=sorted(group0,key=lambda r:r["image_id"])
random.Random(SPLIT_SEED).shuffle(pool)

train_records=pool[:208]
validation_records=pool[208:248]
core_test=pool[248:318]
test_records=core_test+sorted(group1,key=lambda r:r["image_id"])

for split,records in [
    ("train",train_records),
    ("validation",validation_records),
    ("test",test_records),
]:
    for i,r in enumerate(records):
        r["id"]=f"{split}-{i:04d}"
        r["split"]=split

if len(test_records)!=391:
    raise RuntimeError("test gallery must contain exactly 391 photographs")

all_seen={}
for split,records in [("train",train_records),("validation",validation_records),("test",test_records)]:
    for r in records:
        prior=all_seen.get(r["pixel_sha256"])
        if prior and prior!=split:
            raise RuntimeError(f"decoded-pixel leakage across {prior}/{split}")
        all_seen[r["pixel_sha256"]]=split

test_caption_count=sum(len(r["captions"]) for r in test_records)
if test_caption_count!=1737:
    raise RuntimeError(f"expected 1737 test captions, got {test_caption_count}")

print({
    "train":[len(train_records),sum(len(r["captions"]) for r in train_records)],
    "validation":[len(validation_records),sum(len(r["captions"]) for r in validation_records)],
    "test":[len(test_records),test_caption_count],
})

In [ ]:
# @title Dataset provenance
dataset_manifest={
    "source":CORPUS_REPO,
    "revision":CORPUS_REVISION,
    "shard":CORPUS_PATH,
    "shard_bytes":CORPUS_BYTES,
    "shard_sha256":CORPUS_SHA256,
    "split_seed":SPLIT_SEED,
    "roles":{
        "train":[r["image_id"] for r in train_records],
        "validation":[r["image_id"] for r in validation_records],
        "test":[r["image_id"] for r in test_records],
    },
    "counts":{
        "train_images":len(train_records),
        "validation_images":len(validation_records),
        "test_images":len(test_records),
        "test_captions":test_caption_count,
    },
}
dataset_manifest["role_digest"]=hashlib.sha256(
    json.dumps(dataset_manifest["roles"],sort_keys=True,separators=(",",":")).encode()
).hexdigest()

(OUTPUT_ROOT/"data"/"dataset_manifest.json").write_text(
    json.dumps(dataset_manifest,indent=2),encoding="utf-8"
)
print("role_digest",dataset_manifest["role_digest"])

# 5. Common retrieval evaluator

For a score matrix:

```text
shape = [number of images, number of captions]
```

`owners[j]` identifies the photograph owning caption `j`.

Report:

- image→text R@1, R@5, R@10
- text→image R@1, R@5, R@10
- median rank in both directions
- `rsum`

Only ranking matters. Model-specific score scales do not.

In [ ]:
# @title Retrieval metrics
def gallery(records):
    texts=[];owners=[]
    for i,r in enumerate(records):
        for caption in r["captions"]:
            texts.append(str(caption));owners.append(i)
    return texts,owners

def retrieval_metrics(scores,owners):
    grid=np.asarray(scores,dtype=np.float64)
    owners=np.asarray(owners,dtype=int)
    if grid.ndim!=2 or grid.shape[1]!=len(owners):
        raise ValueError("score grid/owner mismatch")
    if set(owners.tolist())!=set(range(grid.shape[0])):
        raise ValueError("every image must own at least one caption")
    if not np.isfinite(grid).all():
        raise ValueError("scores must be finite")

    i2t_ranks=[]
    for i,row in enumerate(grid):
        order=np.argsort(-row,kind="stable")
        positions=np.flatnonzero(owners[order]==i)
        i2t_ranks.append(int(positions[0])+1)

    t2i_ranks=[]
    for j,owner in enumerate(owners):
        order=np.argsort(-grid[:,j],kind="stable")
        position=int(np.flatnonzero(order==owner)[0])+1
        t2i_ranks.append(position)

    out={"n_images":grid.shape[0],"n_captions":grid.shape[1]}
    for k in RECALL_KS:
        out[f"i2t_recall_at_{k}"]=float(np.mean(np.asarray(i2t_ranks)<=k))
        out[f"t2i_recall_at_{k}"]=float(np.mean(np.asarray(t2i_ranks)<=k))
    out["i2t_median_rank"]=float(np.median(i2t_ranks))
    out["t2i_median_rank"]=float(np.median(t2i_ranks))
    out["rsum"]=sum(out[f"{d}_recall_at_{k}"] for d in ("i2t","t2i") for k in RECALL_KS)
    return out

def canonical_caption_indices(owners,n_images):
    owners=np.asarray(owners)
    return [int(np.flatnonzero(owners==i)[0]) for i in range(n_images)]

def candidate_oracle(scores,owners,k):
    grid=np.asarray(scores);owners=np.asarray(owners)
    i2t=[]
    for i,row in enumerate(grid):
        top=np.argsort(-row,kind="stable")[:min(k,grid.shape[1])]
        i2t.append(bool(np.any(owners[top]==i)))
    queries=canonical_caption_indices(owners,grid.shape[0])
    t2i=[]
    for j in queries:
        top=np.argsort(-grid[:,j],kind="stable")[:min(k,grid.shape[0])]
        t2i.append(bool(owners[j] in top))
    return {
        "i2t_candidate_oracle":float(np.mean(i2t)),
        "t2i_candidate_oracle":float(np.mean(t2i)),
    }

# 6. Analytical chance and colour/keyword baselines

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. Record the baseline before interpreting the more complex result.

In [ ]:
# @title Baselines
import re
from math import comb

def expected_recall(n,total_positives,k):
    if k>=n:return 1.0
    if total_positives<=0:return 0.0
    return 1.0 - comb(n-total_positives,k)/comb(n,k)

def chance_baseline(records):
    texts,owners=gallery(records)
    n_images=len(records);n_captions=len(texts)
    out={"n_images":n_images,"n_captions":n_captions}
    for k in RECALL_KS:
        out[f"i2t_recall_at_{k}"]=sum(
            expected_recall(n_captions,len(r["captions"]),min(k,n_captions))
            for r in records
        )/n_images
        out[f"t2i_recall_at_{k}"]=min(k,n_images)/n_images
    out["i2t_median_rank"]=None
    out["t2i_median_rank"]=(n_images+1)/2
    out["rsum"]=sum(out[f"{d}_recall_at_{k}"] for d in ("i2t","t2i") for k in RECALL_KS)
    return out

def colour_signature(path,grid=3):
    with Image.open(path) as im:
        small=im.convert("RGB").resize((grid*8,grid*8),Image.BILINEAR)
        arr=np.asarray(small,dtype=np.float32)/255.0
    out=[]
    for y in range(grid):
        for x in range(grid):
            cell=arr[y*8:(y+1)*8,x*8:(x+1)*8]
            out.extend(cell.mean(axis=(0,1)).tolist())
    return np.asarray(out,dtype=np.float32)

def tokens(text):
    return set(re.findall(r"[a-z0-9]+",text.lower()))

def unigram_f1(a,b):
    aa=tokens(a);bb=tokens(b)
    if not aa or not bb:return 0.0
    overlap=len(aa&bb)
    if not overlap:return 0.0
    p=overlap/len(aa);r=overlap/len(bb)
    return 2*p*r/(p+r)

def colour_keyword_baseline(train,records):
    train_sig=[colour_signature(r["path"]) for r in train]
    train_caps=[(c,i) for i,r in enumerate(train) for c in r["captions"]]
    texts,owners=gallery(records)
    test_sig=[colour_signature(r["path"]) for r in records]

    # T2I score grid.
    t2i=np.zeros((len(records),len(texts)),dtype=np.float32)
    for j,text in enumerate(texts):
        best=max(train_caps,key=lambda x:unigram_f1(text,x[0]))[1]
        for i,sig in enumerate(test_sig):
            t2i[i,j]=-float(np.linalg.norm(sig-train_sig[best]))

    # I2T score grid.
    i2t=np.zeros_like(t2i)
    for i,sig in enumerate(test_sig):
        twin=min(range(len(train)),key=lambda t:float(np.linalg.norm(sig-train_sig[t])))
        twin_caps=train[twin]["captions"]
        for j,text in enumerate(texts):
            i2t[i,j]=max(unigram_f1(text,c) for c in twin_caps)

    mt=retrieval_metrics(t2i,owners)
    mi=retrieval_metrics(i2t,owners)
    out={"n_images":len(records),"n_captions":len(texts)}
    for k in RECALL_KS:
        out[f"i2t_recall_at_{k}"]=mi[f"i2t_recall_at_{k}"]
        out[f"t2i_recall_at_{k}"]=mt[f"t2i_recall_at_{k}"]
    out["i2t_median_rank"]=mi["i2t_median_rank"]
    out["t2i_median_rank"]=mt["t2i_median_rank"]
    out["rsum"]=sum(out[f"{d}_recall_at_{k}"] for d in ("i2t","t2i") for k in RECALL_KS)
    return out

# 7. Stage and verify all three model snapshots

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Stage exact model files
from huggingface_hub import hf_hub_download

def stage_snapshot(spec,root):
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    for name,size,digest in spec["files"]:
        path=Path(hf_hub_download(
            repo_id=spec["model_id"],
            filename=name,
            revision=spec["revision"],
            local_dir=str(root),
        ))
        if path.stat().st_size!=size or sha256_file(path)!=digest:
            raise ValueError(f"{spec['model_id']} {name}: digest/size mismatch")
    return root

SIGLIP2_ROOT=stage_snapshot(SIGLIP2,WORK_ROOT/"models"/"siglip2")
SIGLIP1_ROOT=stage_snapshot(SIGLIP1,WORK_ROOT/"models"/"siglip1")
BLIP_ROOT=stage_snapshot(BLIP,WORK_ROOT/"models"/"blip")

print("All model snapshots verified.")

# 8. Dual-encoder embedding helpers

SigLIP v1 and v2 produce unit-normalized **768-d** embeddings.

BLIP ITC produces unit-normalized **256-d** embeddings.

Embeddings from different models are separate coordinate systems and MUST never be mixed.

In [ ]:
# @title SigLIP embedding wrapper
from transformers import AutoModel, AutoProcessor
import torch.nn.functional as F

def siglip_embed(spec,root,records,texts,image_batch=16,text_batch=64):
    processor=AutoProcessor.from_pretrained(str(root),local_files_only=True,trust_remote_code=False)
    model=AutoModel.from_pretrained(
        str(root),local_files_only=True,trust_remote_code=False
    ).to(DEVICE).eval()

    image_chunks=[]
    t0=time.perf_counter()
    for start in range(0,len(records),image_batch):
        images=[]
        for r in records[start:start+image_batch]:
            with Image.open(r["path"]) as im:
                images.append(im.convert("RGB"))
        batch=processor(images=images,return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            feat=model.get_image_features(**batch)
            feat=F.normalize(feat,p=2,dim=-1)
        image_chunks.append(feat.cpu())
    image_seconds=time.perf_counter()-t0

    text_chunks=[]
    t0=time.perf_counter()
    for start in range(0,len(texts),text_batch):
        chunk=[str(x).lower() for x in texts[start:start+text_batch]]
        batch=processor(
            text=chunk,padding="max_length",max_length=64,return_tensors="pt"
        ).to(DEVICE)
        with torch.inference_mode():
            feat=model.get_text_features(**batch)
            feat=F.normalize(feat,p=2,dim=-1)
        text_chunks.append(feat.cpu())
    text_seconds=time.perf_counter()-t0

    images=torch.cat(image_chunks).numpy().astype(np.float32,copy=False)
    text=torch.cat(text_chunks).numpy().astype(np.float32,copy=False)
    del model,processor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

    return images,text,{
        "image_embedding_seconds":image_seconds,
        "text_embedding_seconds":text_seconds,
    }

In [ ]:
# @title BLIP embedding/reranking wrapper
from transformers import BlipForImageTextRetrieval, BlipProcessor

class BlipWorkshop:
    def __init__(self,root=BLIP_ROOT,device=DEVICE):
        self.root=Path(root)
        self.device=device
        self.processor=BlipProcessor.from_pretrained(str(root),local_files_only=True)
        self.model=BlipForImageTextRetrieval.from_pretrained(
            str(root),
            local_files_only=True,
            trust_remote_code=False,
            use_safetensors=False,
            weights_only=True,
            dtype=torch.float32,
        ).to(device).eval()

    def raw_image_embeds(self,records,batch_size=8):
        out=[]
        t0=time.perf_counter()
        for start in range(0,len(records),batch_size):
            images=[]
            for r in records[start:start+batch_size]:
                with Image.open(r["path"]) as im:
                    images.append(im.convert("RGB"))
            pixels=self.processor(images=images,return_tensors="pt")["pixel_values"].to(self.device)
            with torch.inference_mode():
                embeds=self.model.vision_model(pixel_values=pixels)[0]
            out.extend(e.detach().clone() for e in embeds)
        return out,time.perf_counter()-t0

    def image_features(self,raw_embeds,batch_size=64):
        out=[]
        with torch.inference_mode():
            for start in range(0,len(raw_embeds),batch_size):
                stacked=torch.stack([x[0] for x in raw_embeds[start:start+batch_size]])
                out.append(F.normalize(self.model.vision_proj(stacked),dim=-1).cpu())
        return torch.cat(out).numpy().astype(np.float32,copy=False)

    def text_features(self,texts,batch_size=64):
        out=[];t0=time.perf_counter()
        for start in range(0,len(texts),batch_size):
            tokens=self.processor.tokenizer(
                list(texts[start:start+batch_size]),padding=True,return_tensors="pt"
            ).to(self.device)
            with torch.inference_mode():
                hidden=self.model.text_encoder(
                    input_ids=tokens["input_ids"],attention_mask=tokens["attention_mask"]
                )[0]
                out.append(F.normalize(self.model.text_proj(hidden[:,0,:]),dim=-1).cpu())
        return torch.cat(out).numpy().astype(np.float32,copy=False),time.perf_counter()-t0

    def itm_probabilities(self,pairs,batch_size=16):
        values=[]
        t0=time.perf_counter()
        for start in range(0,len(pairs),batch_size):
            chunk=pairs[start:start+batch_size]
            tokens=self.processor.tokenizer(
                [text for _embed,text in chunk],padding=True,return_tensors="pt"
            ).to(self.device)
            image_embeds=torch.stack([embed for embed,_text in chunk]).to(self.device)
            image_atts=torch.ones(image_embeds.shape[:2],dtype=torch.long,device=self.device)
            with torch.inference_mode():
                hidden=self.model.text_encoder(
                    input_ids=tokens["input_ids"],
                    attention_mask=tokens["attention_mask"],
                    encoder_hidden_states=image_embeds,
                    encoder_attention_mask=image_atts,
                )[0]
                logits=self.model.itm_head(hidden[:,0,:])
                values.extend(torch.softmax(logits.float(),dim=-1)[:,1].cpu().tolist())
        return values,time.perf_counter()-t0

# 9. Validation retrieval

Before the 391-image test gallery is opened, evaluate each frozen coarse retriever on the 40-image validation gallery.

This confirms the common evaluator and makes validation available for the optional BLIP adaptation.

In [ ]:
# @title Validation embeddings and metrics
validation_texts,validation_owners=gallery(validation_records)

validation_scores={}
validation_runtime={}

v_img,v_txt,timing=siglip_embed(
    SIGLIP2,SIGLIP2_ROOT,validation_records,validation_texts
)
validation_scores["siglip2"]=v_img@v_txt.T
validation_runtime["siglip2"]=timing
del v_img,v_txt

v_img,v_txt,timing=siglip_embed(
    SIGLIP1,SIGLIP1_ROOT,validation_records,validation_texts
)
validation_scores["siglip1"]=v_img@v_txt.T
validation_runtime["siglip1"]=timing
del v_img,v_txt

blip_val=BlipWorkshop()
val_raw,val_image_seconds=blip_val.raw_image_embeds(validation_records)
val_img=blip_val.image_features(val_raw)
val_txt,val_text_seconds=blip_val.text_features(validation_texts)
validation_scores["blip_itc"]=val_img@val_txt.T
validation_runtime["blip_itc"]={
    "image_embedding_seconds":val_image_seconds,
    "text_embedding_seconds":val_text_seconds,
}

validation_rows=[]
for name,scores in validation_scores.items():
    validation_rows.append({"model":name,**retrieval_metrics(scores,validation_owners)})
validation_table=pd.DataFrame(validation_rows)
display(validation_table)
validation_table.to_csv(OUTPUT_ROOT/"validation"/"retrieval_metrics.csv",index=False)

# 10. Validation reranking sanity

Use the fixed canonical:

```text
RERANK_TOP_K = 5
```

The correct item must already be inside the shortlist; the reranker cannot recover candidates the first-stage retriever never supplied.

In [ ]:
# @title Generic BLIP ITM reranker over arbitrary coarse score grids
def rerank_with_blip(blip,raw_image_embeds,texts,owners,coarse,k):
    owners=np.asarray(owners,dtype=int)
    n_images=coarse.shape[0]

    # Image -> text
    pairs=[];pair_index=[]
    for i in range(n_images):
        top=np.argsort(-coarse[i],kind="stable")[:min(k,coarse.shape[1])]
        for j in top:
            pairs.append((raw_image_embeds[i],texts[int(j)]))
            pair_index.append((i,int(j)))
    probs,seconds_i2t=blip.itm_probabilities(pairs)
    best={}
    for (i,j),p in zip(pair_index,probs):
        if i not in best or p>best[i][0]:
            best[i]=(p,j)
    i2t_r1=float(np.mean([owners[j]==i for i,(_p,j) in best.items()]))

    # Canonical caption -> image
    queries=canonical_caption_indices(owners,n_images)
    pairs=[];pair_index=[]
    for j in queries:
        top=np.argsort(-coarse[:,j],kind="stable")[:min(k,n_images)]
        for i in top:
            pairs.append((raw_image_embeds[int(i)],texts[j]))
            pair_index.append((int(i),j))
    probs2,seconds_t2i=blip.itm_probabilities(pairs)
    best2={}
    for (i,j),p in zip(pair_index,probs2):
        if j not in best2 or p>best2[j][0]:
            best2[j]=(p,i)
    t2i_r1=float(np.mean([owners[j]==i for j,(_p,i) in best2.items()]))

    # Hard-negative pair accuracy.
    pairs=[]
    for i,j_pos in enumerate(queries):
        row=coarse[i].copy()
        row[owners==i]=-np.inf
        j_neg=int(np.argmax(row))
        pairs.append((raw_image_embeds[i],texts[j_pos]))
        pairs.append((raw_image_embeds[i],texts[j_neg]))
    probs3,seconds_pair=blip.itm_probabilities(pairs)
    pair_accuracy=float(np.mean(np.asarray(probs3[0::2])>np.asarray(probs3[1::2])))

    return {
        "rerank_top_k":k,
        "itm_i2t_recall_at_1":i2t_r1,
        "itm_t2i_recall_at_1":t2i_r1,
        "itm_pair_accuracy":pair_accuracy,
        "pair_evaluations":len(probs)+len(probs2)+len(probs3),
        "seconds":seconds_i2t+seconds_t2i+seconds_pair,
        **candidate_oracle(coarse,owners,k),
    }

validation_rerank=[]
for name,scores in validation_scores.items():
    result=rerank_with_blip(
        blip_val,val_raw,validation_texts,validation_owners,scores,RERANK_TOP_K
    )
    validation_rerank.append({"candidate_source":name,**result})
validation_rerank_table=pd.DataFrame(validation_rerank)
display(validation_rerank_table)
validation_rerank_table.to_csv(OUTPUT_ROOT/"validation"/"reranking.csv",index=False)

# 11. `FULL` — BLIP retrieval adaptation

Only BLIP is adapted because its live DIMER contract is already an image-caption retrieval adaptation.

Trainable:

- last two fused text-encoder blocks;
- `vision_proj`;
- `text_proj`;
- `itm_head`.

Frozen:

- complete vision encoder;
- text embeddings;
- earlier text blocks.

Objective:

```text
ITC contrastive loss
+
ITM hard-negative matching loss
```

Selection:

> highest validation `rsum`

Epoch 0 remains the frozen baseline.

In [ ]:
# @title BLIP adaptation helpers
from safetensors.torch import save_file, load_file
import math

ITC_TEMPERATURE=0.07

def blip_trainable_names(model,n_layers=2):
    total=len(model.text_encoder.encoder.layer)
    first=total-n_layers
    prefixes=tuple(f"text_encoder.encoder.layer.{k}." for k in range(first,total))+(
        "vision_proj.","text_proj.","itm_head.",
    )
    return [name for name,_ in model.named_parameters() if name.startswith(prefixes)]

def blip_score_from_raw(blip,records,raw_embeds):
    texts,owners=gallery(records)
    image_feat=blip.image_features(raw_embeds)
    text_feat,_=blip.text_features(texts)
    scores=image_feat@text_feat.T
    return retrieval_metrics(scores,owners),scores,texts,owners

def adapt_blip(blip,train,val,epochs=4,lr=2e-5,batch_size=16,n_layers=2,seed=0):
    model=blip.model
    names=blip_trainable_names(model,n_layers)
    wanted=set(names)
    for name,param in model.named_parameters():
        param.requires_grad_(name in wanted)
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=0.01)

    train_raw,_=blip.raw_image_embeds(train)
    val_raw,_=blip.raw_image_embeds(val)

    pairs=[(i,str(c)) for i,r in enumerate(train) for c in r["captions"]]
    generator=torch.Generator().manual_seed(seed)

    def score_val():
        model.eval()
        metrics,_,_,_=blip_score_from_raw(blip,val,val_raw)
        return metrics

    initial={k:v.detach().clone() for k,v in model.state_dict().items() if k in wanted}
    best={k:v.clone() for k,v in initial.items()}
    history=[{"epoch":0,"train_loss":None,"val":score_val(),"note":"frozen model"}]
    best_epoch=0
    best_score=history[0]["val"]["rsum"]

    t0=time.perf_counter()
    try:
        for epoch in range(1,epochs+1):
            model.train()
            order=torch.randperm(len(pairs),generator=generator).tolist()
            losses=[]
            for start in range(0,len(order),batch_size):
                chosen=[pairs[j] for j in order[start:start+batch_size]]
                if len(chosen)<2:
                    continue

                image_index=torch.tensor([i for i,_ in chosen],device=DEVICE)
                image_embeds=torch.stack([train_raw[i] for i,_ in chosen]).to(DEVICE)
                tokens=blip.processor.tokenizer(
                    [c for _,c in chosen],padding=True,return_tensors="pt"
                ).to(DEVICE)

                text_hidden=model.text_encoder(
                    input_ids=tokens["input_ids"],
                    attention_mask=tokens["attention_mask"],
                )[0]
                text_feat=F.normalize(model.text_proj(text_hidden[:,0,:]),dim=-1)
                image_feat=F.normalize(model.vision_proj(image_embeds[:,0,:]),dim=-1)
                sim=image_feat@text_feat.T/ITC_TEMPERATURE

                same=image_index[:,None].eq(image_index[None,:])
                targets=same.float()
                targets=targets/targets.sum(dim=1,keepdim=True)
                log_i=torch.log_softmax(sim,dim=1)
                log_t=torch.log_softmax(sim.T,dim=1)
                loss_itc=-((targets*log_i).sum(1).mean()+(targets.T*log_t).sum(1).mean())/2

                with torch.no_grad():
                    w_text=torch.softmax(sim.detach().float(),dim=1)
                    w_text=w_text.masked_fill(same,0.0)
                    w_text=w_text/(w_text.sum(dim=1,keepdim=True)+1e-12)

                    w_image=torch.softmax(sim.detach().float().T,dim=1)
                    w_image=w_image.masked_fill(same.T,0.0)
                    w_image=w_image/(w_image.sum(dim=1,keepdim=True)+1e-12)

                    # Every normal batch contains at least two photographs. Refuse pathological batches.
                    if torch.any(w_text.sum(dim=1)<=0) or torch.any(w_image.sum(dim=1)<=0):
                        raise RuntimeError("BLIP batch lacks a valid cross-photograph hard negative")
                    neg_text=torch.multinomial(w_text,1).squeeze(1)
                    neg_image=torch.multinomial(w_image,1).squeeze(1)

                itm_images=torch.cat([image_embeds,image_embeds,image_embeds[neg_image]])
                itm_ids=torch.cat([
                    tokens["input_ids"],
                    tokens["input_ids"][neg_text],
                    tokens["input_ids"],
                ])
                itm_mask=torch.cat([
                    tokens["attention_mask"],
                    tokens["attention_mask"][neg_text],
                    tokens["attention_mask"],
                ])
                image_atts=torch.ones(itm_images.shape[:2],dtype=torch.long,device=DEVICE)

                fused=model.text_encoder(
                    input_ids=itm_ids,
                    attention_mask=itm_mask,
                    encoder_hidden_states=itm_images,
                    encoder_attention_mask=image_atts,
                )[0]
                logits=model.itm_head(fused[:,0,:])
                labels=torch.cat([
                    torch.ones(len(chosen),dtype=torch.long,device=DEVICE),
                    torch.zeros(2*len(chosen),dtype=torch.long,device=DEVICE),
                ])
                loss_itm=F.cross_entropy(logits,labels)
                loss=loss_itc+loss_itm

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params,1.0)
                optimizer.step()
                losses.append(float(loss.detach()))

            model.eval()
            row={
                "epoch":epoch,
                "train_loss":float(np.mean(losses)),
                "val":score_val(),
            }
            history.append(row)
            print(row)
            if row["val"]["rsum"]>best_score:
                best_score=row["val"]["rsum"]
                best_epoch=epoch
                best={k:v.detach().clone() for k,v in model.state_dict().items() if k in wanted}
    except BaseException:
        merged=dict(model.state_dict());merged.update(initial)
        model.load_state_dict(merged,strict=True)
        model.eval()
        for p in model.parameters():p.requires_grad_(False)
        raise

    merged=dict(model.state_dict());merged.update(best)
    model.load_state_dict(merged,strict=True)
    model.eval()
    for p in model.parameters():p.requires_grad_(False)

    return {
        "trainable_names":names,
        "n_trainable":sum(v.numel() for n,v in model.named_parameters() if n in wanted),
        "n_total":sum(p.numel() for p in model.parameters()),
        "epochs":epochs,
        "best_epoch":best_epoch,
        "selection":"highest validation rsum",
        "lr":lr,
        "batch_size":batch_size,
        "itc_temperature":ITC_TEMPERATURE,
        "history":history,
        "seconds":time.perf_counter()-t0,
    }

In [ ]:
# @title Optional FULL BLIP adaptation and artifact export
blip_adapter_report=None
blip_artifact_manifest=None

if WORKSHOP_TIER=="FULL":
    blip_adapter_report=adapt_blip(
        blip_val,
        train_records,
        validation_records,
        epochs=BLIP_EPOCHS,
        lr=BLIP_LEARNING_RATE,
        batch_size=BLIP_BATCH_SIZE,
        n_layers=BLIP_TRAINABLE_TEXT_LAYERS,
        seed=SEED,
    )

    artifact_dir=OUTPUT_ROOT/"artifacts"/"blip"
    names=blip_adapter_report["trainable_names"]
    state=blip_val.model.state_dict()
    tensors={name:state[name].detach().cpu().contiguous() for name in names}
    weights_path=artifact_dir/"adapter.safetensors"
    save_file(tensors,str(weights_path),metadata={"format":"pt"})

    blip_artifact_manifest={
        "format":"org.valcorza.blip-itm-base-coco.adapter.v1",
        "format_version":"1.0",
        "base":{
            "model_id":BLIP["model_id"],
            "revision":BLIP["revision"],
            "weight_file":"pytorch_model.bin",
            "weight_sha256":"017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f",
        },
        "adapter":{
            k:v for k,v in blip_adapter_report.items()
            if k not in ("trainable_names","history")
        },
        "history":blip_adapter_report["history"],
        "tensors":names,
        "files":[{
            "path":"adapter.safetensors",
            "bytes":weights_path.stat().st_size,
            "sha256":sha256_file(weights_path),
        }],
    }
    (artifact_dir/"manifest.json").write_text(
        json.dumps(blip_artifact_manifest,indent=2,default=str),encoding="utf-8"
    )
    (OUTPUT_ROOT/"adaptation"/"blip"/"training_history.json").write_text(
        json.dumps(blip_adapter_report,indent=2,default=str),encoding="utf-8"
    )
    print({
        "best_epoch":blip_adapter_report["best_epoch"],
        "adapter_bytes":weights_path.stat().st_size,
    })
else:
    print("STANDARD: BLIP adaptation skipped.")

# 12. Freeze-before-test

The following are now fixed:

- split membership;
- 391-image test gallery;
- retrieval Recall@K values;
- canonical first-caption rule for text→image reranking;
- `RERANK_TOP_K`;
- embedding normalization;
- BLIP selected epoch/artifact in `FULL`;
- common metric implementation.

Only after writing this record do we compute the final test embeddings/ranks.

In [ ]:
# @title Freeze experiment
frozen={
    "notebook_spec":"2.1",
    "profile":"MULTI-CAPABILITY",
    "mode":"WORKSHOP",
    "tier":WORKSHOP_TIER,
    "dataset":dataset_manifest,
    "models":{
        "siglip2":{
            "id":SIGLIP2["model_id"],"revision":SIGLIP2["revision"],
            "weight_sha256":"612923381c76ec5a9bed335d1c48827e3f2e506ac31b044b63b2031fadee6a0b",
            "embedding_dim":768,
        },
        "siglip1":{
            "id":SIGLIP1["model_id"],"revision":SIGLIP1["revision"],
            "weight_sha256":"f0cee7c815135c44a515eff72ab3040499744920442bc25567cd04efc93f8f65",
            "embedding_dim":768,
        },
        "blip":{
            "id":BLIP["model_id"],"revision":BLIP["revision"],
            "weight_sha256":"017fb3e7f4e125f13a8a4717f1402dbe0d0bb877474b4a203db13a4447b0227f",
            "embedding_dim":256,
        },
    },
    "evaluation":{
        "recall_k":list(RECALL_KS),
        "rerank_top_k":RERANK_TOP_K,
        "query_caption_rule":"first caption per photograph",
        "gallery_sizes":list(GALLERY_SIZES),
        "normalization":"L2 unit norm; cosine via dot product",
    },
    "blip_adapter":blip_artifact_manifest,
}
freeze_path=OUTPUT_ROOT/"frozen"/"frozen_experiment.json"
freeze_path.write_text(json.dumps(frozen,indent=2,default=str),encoding="utf-8")
print("Frozen:",freeze_path)

# 13. Final test gallery — analytical and non-neural baselines

In [ ]:
# @title Test baselines
test_texts,test_owners=gallery(test_records)

chance=chance_baseline(test_records)
colour=colour_keyword_baseline(train_records,test_records)

display(pd.DataFrame([
    {"model":"chance",**chance},
    {"model":"colour_keyword",**colour},
]))

# 14. SigLIP 2 test index

In [ ]:
# @title SigLIP 2 test embeddings and index export
siglip2_img,siglip2_txt,siglip2_timing=siglip_embed(
    SIGLIP2,SIGLIP2_ROOT,test_records,test_texts
)
siglip2_scores=siglip2_img@siglip2_txt.T
siglip2_metrics=retrieval_metrics(siglip2_scores,test_owners)

np.save(OUTPUT_ROOT/"index"/"siglip2_images.npy",siglip2_img)
np.save(OUTPUT_ROOT/"index"/"siglip2_texts.npy",siglip2_txt)

print(siglip2_metrics)

# 15. SigLIP v1 test index

In [ ]:
# @title SigLIP v1 test embeddings and index export
siglip1_img,siglip1_txt,siglip1_timing=siglip_embed(
    SIGLIP1,SIGLIP1_ROOT,test_records,test_texts
)
siglip1_scores=siglip1_img@siglip1_txt.T
siglip1_metrics=retrieval_metrics(siglip1_scores,test_owners)

np.save(OUTPUT_ROOT/"index"/"siglip1_images.npy",siglip1_img)
np.save(OUTPUT_ROOT/"index"/"siglip1_texts.npy",siglip1_txt)

print(siglip1_metrics)

# 16. Frozen BLIP ITC test index

In [ ]:
# @title Fresh frozen BLIP test embeddings
# Dispose validation/adapted object before test reconstruction.
del blip_val
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

blip_frozen=BlipWorkshop()
blip_raw,blip_image_seconds=blip_frozen.raw_image_embeds(test_records)
blip_img=blip_frozen.image_features(blip_raw)
blip_txt,blip_text_seconds=blip_frozen.text_features(test_texts)
blip_scores=blip_img@blip_txt.T
blip_metrics=retrieval_metrics(blip_scores,test_owners)

np.save(OUTPUT_ROOT/"index"/"blip_itc_images.npy",blip_img)
np.save(OUTPUT_ROOT/"index"/"blip_itc_texts.npy",blip_txt)

print(blip_metrics)

# 17. Principal frozen retrieval table

In [ ]:
# @title Retrieval comparison
retrieval_rows=[
    {"model":"chance","dim":None,**chance},
    {"model":"colour_keyword","dim":None,**colour},
    {"model":"siglip_v1","dim":768,**siglip1_metrics},
    {"model":"siglip2","dim":768,**siglip2_metrics},
    {"model":"blip_itc","dim":256,**blip_metrics},
]
retrieval_table=pd.DataFrame(retrieval_rows)
display(retrieval_table)
retrieval_table.to_csv(OUTPUT_ROOT/"test"/"retrieval_metrics.csv",index=False)

# 18. BLIP ITM reranks all three first-stage systems

The same frozen BLIP ITM head reranks:

- SigLIP v1 candidates;
- SigLIP 2 candidates;
- BLIP ITC candidates.

This isolates the effect of the **candidate generator** while holding the expensive second stage fixed.

In [ ]:
# @title Frozen BLIP ITM reranking
rerank_rows=[]
for name,scores in [
    ("siglip_v1",siglip1_scores),
    ("siglip2",siglip2_scores),
    ("blip_itc",blip_scores),
]:
    result=rerank_with_blip(
        blip_frozen,blip_raw,test_texts,test_owners,scores,RERANK_TOP_K
    )
    coarse=retrieval_metrics(scores,test_owners)
    rerank_rows.append({
        "candidate_source":name,
        "coarse_i2t_r1":coarse["i2t_recall_at_1"],
        "coarse_t2i_r1":coarse["t2i_recall_at_1"],
        **result,
    })

rerank_table=pd.DataFrame(rerank_rows)
display(rerank_table)
rerank_table.to_csv(OUTPUT_ROOT/"test"/"reranking_metrics.csv",index=False)

# 19. `FULL` — adapted BLIP coarse retrieval and reranking

Reload the adapter onto a **fresh** pinned BLIP base.

Then measure:

1. adapted BLIP ITC as its own first-stage retriever;
2. adapted BLIP ITM reranking adapted BLIP ITC;
3. adapted BLIP ITM reranking the unchanged frozen SigLIP 2 shortlist.

The third experiment is the most deployment-like:

```text
frozen scalable vector index
→ domain-adapted expensive reranker
```

In [ ]:
# @title FULL fresh BLIP reload and test
adapted_blip_metrics=None
adapted_rerank_rows=[]
blip_reload_parity=None

if WORKSHOP_TIER=="FULL":
    fresh=BlipWorkshop()
    manifest=json.loads((OUTPUT_ROOT/"artifacts"/"blip"/"manifest.json").read_text())
    weights_path=OUTPUT_ROOT/"artifacts"/"blip"/manifest["files"][0]["path"]
    entry=manifest["files"][0]
    if weights_path.stat().st_size!=entry["bytes"] or sha256_file(weights_path)!=entry["sha256"]:
        raise ValueError("BLIP adapter digest/size mismatch")
    tensors=load_file(str(weights_path))
    if sorted(tensors)!=sorted(manifest["tensors"]):
        raise ValueError("BLIP adapter tensor list mismatch")
    state=fresh.model.state_dict()
    merged=dict(state)
    for name,value in tensors.items():
        if tuple(value.shape)!=tuple(state[name].shape):
            raise ValueError(f"adapter shape mismatch: {name}")
        merged[name]=value.to(state[name].dtype)
    fresh.model.load_state_dict(merged,strict=True)
    fresh.model.eval()

    adapted_raw,adapted_image_seconds=fresh.raw_image_embeds(test_records)
    adapted_img=fresh.image_features(adapted_raw)
    adapted_txt,adapted_text_seconds=fresh.text_features(test_texts)
    adapted_scores=adapted_img@adapted_txt.T
    adapted_blip_metrics=retrieval_metrics(adapted_scores,test_owners)

    for source,scores in [
        ("adapted_blip_itc",adapted_scores),
        ("frozen_siglip2",siglip2_scores),
    ]:
        result=rerank_with_blip(
            fresh,adapted_raw,test_texts,test_owners,scores,RERANK_TOP_K
        )
        adapted_rerank_rows.append({"candidate_source":source,**result})

    # Adapter reload parity: deterministic projection rows on fixed raw embeddings/texts.
    repeat_img=fresh.image_features(adapted_raw[:8])
    repeat_txt,_=fresh.text_features(test_texts[:8])
    blip_reload_parity={
        "image_finite":bool(np.isfinite(repeat_img).all()),
        "text_finite":bool(np.isfinite(repeat_txt).all()),
        "adapter_sha256":entry["sha256"],
        "tensors":len(tensors),
    }

    print("Adapted BLIP:",adapted_blip_metrics)
    display(pd.DataFrame(adapted_rerank_rows))
else:
    print("STANDARD: adapted BLIP test skipped.")

# 20. Gallery-size sensitivity

In [ ]:
# @title Nested gallery sizes
def subset_gallery(records,scores,n_images):
    sub_records=records[:n_images]
    texts,owners=gallery(sub_records)
    caption_count=len(texts)
    # Because captions are flattened in record order, the first N images' captions form a prefix.
    return scores[:n_images,:caption_count],owners,caption_count

gallery_rows=[]
for n in GALLERY_SIZES:
    for name,scores in [
        ("siglip_v1",siglip1_scores),
        ("siglip2",siglip2_scores),
        ("blip_itc",blip_scores),
    ]:
        sub,owners,n_caps=subset_gallery(test_records,scores,n)
        m=retrieval_metrics(sub,owners)
        gallery_rows.append({
            "gallery_images":n,"captions":n_caps,"model":name,
            "i2t_r1":m["i2t_recall_at_1"],
            "t2i_r1":m["t2i_recall_at_1"],
            "rsum":m["rsum"],
        })
gallery_table=pd.DataFrame(gallery_rows)
display(gallery_table)
gallery_table.to_csv(OUTPUT_ROOT/"test"/"gallery_size_metrics.csv",index=False)

# 21. Category and caption-length diagnostics

VizWiz marks whether text was detected in the source photograph.

The notebook also groups captions by length using cut points learned from **training captions only**.

In [ ]:
# @title Category and caption-length metrics
category_rows=[]
for model,scores in [
    ("siglip_v1",siglip1_scores),
    ("siglip2",siglip2_scores),
    ("blip_itc",blip_scores),
]:
    for category in sorted({r["category"] for r in test_records}):
        image_indices=[i for i,r in enumerate(test_records) if r["category"]==category]
        members=set(image_indices)
        text_indices=[j for j,o in enumerate(test_owners) if o in members]
        remap={old:new for new,old in enumerate(image_indices)}
        sub=scores[np.ix_(image_indices,text_indices)]
        owners=[remap[test_owners[j]] for j in text_indices]
        m=retrieval_metrics(sub,owners)
        category_rows.append({
            "model":model,"category":category,"n_images":len(image_indices),
            "i2t_r1":m["i2t_recall_at_1"],"t2i_r1":m["t2i_recall_at_1"],"rsum":m["rsum"]
        })
category_table=pd.DataFrame(category_rows)
display(category_table)
category_table.to_csv(OUTPUT_ROOT/"test"/"category_metrics.csv",index=False)

train_lengths=[len(c.split()) for r in train_records for c in r["captions"]]
q1,q2=np.quantile(train_lengths,[1/3,2/3])

def length_group(text):
    n=len(text.split())
    return "short" if n<=q1 else "medium" if n<=q2 else "long"

length_rows=[]
owners_arr=np.asarray(test_owners)
for model,scores in [
    ("siglip_v1",siglip1_scores),
    ("siglip2",siglip2_scores),
    ("blip_itc",blip_scores),
]:
    for group in ("short","medium","long"):
        indices=[j for j,t in enumerate(test_texts) if length_group(t)==group]
        ranks=[]
        for j in indices:
            order=np.argsort(-scores[:,j],kind="stable")
            ranks.append(int(np.flatnonzero(order==owners_arr[j])[0])+1)
        length_rows.append({
            "model":model,"caption_length":group,"n":len(indices),
            "t2i_r1":float(np.mean(np.asarray(ranks)==1)),
            "t2i_median_rank":float(np.median(ranks)),
        })
length_table=pd.DataFrame(length_rows)
display(length_table)
length_table.to_csv(OUTPUT_ROOT/"test"/"caption_length_metrics.csv",index=False)

# 22. Hard negatives and deterministic disagreement gallery

In [ ]:
# @title Rank diagnostics
def image_best_correct_and_wrong(scores,records,texts,owners):
    owners=np.asarray(owners)
    rows=[]
    for i,row in enumerate(scores):
        correct=np.flatnonzero(owners==i)
        wrong=np.flatnonzero(owners!=i)
        best_correct=int(correct[np.argmax(row[correct])])
        best_wrong=int(wrong[np.argmax(row[wrong])])
        order=np.argsort(-row,kind="stable")
        correct_rank=int(np.flatnonzero(order==best_correct)[0])+1
        rows.append({
            "image_index":i,
            "image_id":records[i]["image_id"],
            "correct_rank":correct_rank,
            "best_correct_caption":texts[best_correct],
            "best_wrong_caption":texts[best_wrong],
            "correct_score":float(row[best_correct]),
            "wrong_score":float(row[best_wrong]),
            "margin":float(row[best_correct]-row[best_wrong]),
        })
    return pd.DataFrame(rows)

hard_tables={
    "siglip_v1":image_best_correct_and_wrong(siglip1_scores,test_records,test_texts,test_owners),
    "siglip2":image_best_correct_and_wrong(siglip2_scores,test_records,test_texts,test_owners),
    "blip_itc":image_best_correct_and_wrong(blip_scores,test_records,test_texts,test_owners),
}

hard_tables["siglip2"].to_csv(OUTPUT_ROOT/"test"/"hard_negatives.csv",index=False)

rank_compare=hard_tables["siglip2"][["image_index","correct_rank"]].rename(
    columns={"correct_rank":"siglip2_rank"}
).merge(
    hard_tables["siglip_v1"][["image_index","correct_rank"]].rename(
        columns={"correct_rank":"siglip1_rank"}
    ),
    on="image_index"
).merge(
    hard_tables["blip_itc"][["image_index","correct_rank"]].rename(
        columns={"correct_rank":"blip_rank"}
    ),
    on="image_index"
)

rank_compare["siglip2_vs_v1"]=rank_compare["siglip1_rank"]-rank_compare["siglip2_rank"]
rank_compare["siglip2_vs_blip"]=rank_compare["blip_rank"]-rank_compare["siglip2_rank"]
display(rank_compare.sort_values("siglip2_vs_v1",ascending=False).head())

In [ ]:
# @title Show deterministic retrieval disagreement examples
chosen=[
    ("SigLIP2 advantage over v1",int(rank_compare.loc[rank_compare["siglip2_vs_v1"].idxmax(),"image_index"])),
    ("SigLIP v1 advantage over SigLIP2",int(rank_compare.loc[rank_compare["siglip2_vs_v1"].idxmin(),"image_index"])),
    ("SigLIP2 advantage over BLIP ITC",int(rank_compare.loc[rank_compare["siglip2_vs_blip"].idxmax(),"image_index"])),
]

for title,i in chosen:
    r=test_records[i]
    with Image.open(r["path"]) as im:
        fig,ax=plt.subplots(figsize=(7,5))
        ax.imshow(im.convert("RGB"))
        ax.axis("off")
        ax.set_title(title)
        text=(
            f"Gold: {r['captions'][0]}\n\n"
            f"SigLIP2 wrong: {hard_tables['siglip2'].iloc[i]['best_wrong_caption']}\n"
            f"SigLIP v1 wrong: {hard_tables['siglip_v1'].iloc[i]['best_wrong_caption']}\n"
            f"BLIP ITC wrong: {hard_tables['blip_itc'].iloc[i]['best_wrong_caption']}"
        )
        fig.text(0.02,0.01,text,fontsize=8,va="bottom")
        plt.tight_layout(rect=[0,0.17,1,1])
        safe=title.lower().replace(" ","_")
        plt.savefig(OUTPUT_ROOT/"figures"/f"{safe}.png",dpi=150,bbox_inches="tight")
        plt.show()

# 23. Index storage and computation

Raw float32 vector storage:

- SigLIP: `768 × 4 = 3,072 bytes/item`
- BLIP ITC: `256 × 4 = 1,024 bytes/item`

Illustrative one-million-image raw-vector footprint:

- SigLIP ≈ **3.07 GB**
- BLIP ITC ≈ **1.02 GB**

These exclude ANN/vector-database index overhead.

In [ ]:
# @title Runtime and storage table
storage_rows=[
    {
        "model":"siglip_v1","dim":768,
        "bytes_per_float32_vector":768*4,
        "one_million_vectors_GB":768*4*1_000_000/1e9,
        **siglip1_timing,
    },
    {
        "model":"siglip2","dim":768,
        "bytes_per_float32_vector":768*4,
        "one_million_vectors_GB":768*4*1_000_000/1e9,
        **siglip2_timing,
    },
    {
        "model":"blip_itc","dim":256,
        "bytes_per_float32_vector":256*4,
        "one_million_vectors_GB":256*4*1_000_000/1e9,
        "image_embedding_seconds":blip_image_seconds,
        "text_embedding_seconds":blip_text_seconds,
    },
]
storage_table=pd.DataFrame(storage_rows)
display(storage_table)
storage_table.to_csv(OUTPUT_ROOT/"test"/"runtime_storage.csv",index=False)

# Dense similarity search cost for one text query across the 391 image vectors.
for name,img,txt in [
    ("siglip_v1",siglip1_img,siglip1_txt),
    ("siglip2",siglip2_img,siglip2_txt),
    ("blip_itc",blip_img,blip_txt),
]:
    q=txt[0]
    trials=[]
    for _ in range(100):
        t0=time.perf_counter()
        _=img@q
        trials.append(time.perf_counter()-t0)
    print(name,"median matrix-search ms/query",1000*np.median(trials))

# 24. Export and verify embedding indexes

In [ ]:
# @title Index manifest and reload ranking parity
image_ids=[r["image_id"] for r in test_records]
caption_ids=[]
caption_owners=[]
for i,r in enumerate(test_records):
    for j,_caption in enumerate(r["captions"]):
        caption_ids.append(f"{r['image_id']}:{j}")
        caption_owners.append(i)

(OUTPUT_ROOT/"index"/"image_ids.json").write_text(json.dumps(image_ids,indent=2),encoding="utf-8")
(OUTPUT_ROOT/"index"/"caption_ids.json").write_text(json.dumps(caption_ids,indent=2),encoding="utf-8")
(OUTPUT_ROOT/"index"/"caption_owners.json").write_text(json.dumps(caption_owners),encoding="utf-8")

index_manifest={
    "item_order":{
        "images":"image_ids.json",
        "captions":"caption_ids.json",
        "owners":"caption_owners.json",
    },
    "models":{},
}
for name,spec,img_file,txt_file,dim in [
    ("siglip2",SIGLIP2,"siglip2_images.npy","siglip2_texts.npy",768),
    ("siglip1",SIGLIP1,"siglip1_images.npy","siglip1_texts.npy",768),
    ("blip_itc",BLIP,"blip_itc_images.npy","blip_itc_texts.npy",256),
]:
    index_manifest["models"][name]={
        "model_id":spec["model_id"],
        "revision":spec["revision"],
        "embedding_dim":dim,
        "normalization":"L2",
        "dtype":"float32",
        "image_file":{
            "path":img_file,
            "bytes":(OUTPUT_ROOT/"index"/img_file).stat().st_size,
            "sha256":sha256_file(OUTPUT_ROOT/"index"/img_file),
        },
        "text_file":{
            "path":txt_file,
            "bytes":(OUTPUT_ROOT/"index"/txt_file).stat().st_size,
            "sha256":sha256_file(OUTPUT_ROOT/"index"/txt_file),
        },
    }

(OUTPUT_ROOT/"index"/"index_manifest.json").write_text(
    json.dumps(index_manifest,indent=2),encoding="utf-8"
)

# Fresh-boundary ranking check.
for name,img_file,txt_file,reference in [
    ("siglip2","siglip2_images.npy","siglip2_texts.npy",siglip2_scores),
    ("siglip1","siglip1_images.npy","siglip1_texts.npy",siglip1_scores),
    ("blip_itc","blip_itc_images.npy","blip_itc_texts.npy",blip_scores),
]:
    img=np.load(OUTPUT_ROOT/"index"/img_file)
    txt=np.load(OUTPUT_ROOT/"index"/txt_file)
    replay=img[:8]@txt[:64].T
    expected=reference[:8,:64]
    if not np.array_equal(np.argsort(-replay,axis=1),np.argsort(-expected,axis=1)):
        raise RuntimeError(f"{name}: index reload ranking parity failed")

print("Index reload ranking parity: PASS")

# 25. Caption perturbation and no-match behavior

These are **validation/interpretation experiments**, not test tuning.

Retrievers always produce a nearest candidate—even for a query that has no correct photograph in the gallery.

That behavior must not be confused with abstention.

In [ ]:
# @title No-match nearest-neighbour examples using SigLIP2 test index
irrelevant_queries=[
    "a satellite orbiting mars",
    "an underwater coral reef",
]

processor=AutoProcessor.from_pretrained(str(SIGLIP2_ROOT),local_files_only=True)
model=AutoModel.from_pretrained(str(SIGLIP2_ROOT),local_files_only=True).to(DEVICE).eval()
batch=processor(
    text=[q.lower() for q in irrelevant_queries],
    padding="max_length",max_length=64,return_tensors="pt"
).to(DEVICE)
with torch.inference_mode():
    qfeat=F.normalize(model.get_text_features(**batch),dim=-1).cpu().numpy()
del model,processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

for query,feat in zip(irrelevant_queries,qfeat):
    scores=siglip2_img@feat
    top=np.argsort(-scores)[:3]
    print("\nQuery:",query)
    print("Evaluation status: no ground-truth match — nearest-neighbor behavior only")
    for i in top:
        print(float(scores[i]),test_records[int(i)]["captions"][0])

# 26. BYOD contract

Preferred archive:

```text
dataset.zip
├── records.jsonl
└── images/
```

Each JSONL row:

```json
{
  "id": "img001",
  "file": "images/img001.jpg",
  "captions": [
    "a red cup on a table",
    "a cup beside a plate"
  ],
  "split": "test",
  "category": "optional"
}
```

`STANDARD` can run with only an evaluation/test gallery.

`FULL` requires explicit:

- train
- validation
- test

roles if BLIP adaptation is requested.

This notebook is an **in-memory retrieval reference**, not a production vector database.

In [ ]:
# @title Optional BYOD loader and structural validation
import zipfile
import shutil

def safe_extract_zip(path,destination):
    destination=Path(destination).resolve()
    if destination.exists():shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with zipfile.ZipFile(path) as archive:
        infos=archive.infolist()
        if sum(i.file_size for i in infos)>2*1024**3:
            raise ValueError("expanded BYOD ZIP exceeds 2 GiB")
        for info in infos:
            rel=Path(info.filename)
            if rel.is_absolute() or ".." in rel.parts:
                raise ValueError(f"unsafe ZIP path: {info.filename}")
            mode=info.external_attr>>16
            if mode and (mode&0o170000)==0o120000:
                raise ValueError(f"symlink refused: {info.filename}")
        archive.extractall(destination)
    return destination

def load_byod(path):
    root=safe_extract_zip(path,WORK_ROOT/"byod")
    files=list(root.rglob("records.jsonl"))
    if len(files)!=1:
        raise ValueError("BYOD archive needs exactly one records.jsonl")
    base=files[0].parent
    records=[]
    seen=set()
    for line_number,line in enumerate(files[0].read_text(encoding="utf-8").splitlines(),start=1):
        if not line.strip():continue
        row=json.loads(line)
        rid=str(row.get("id",""))
        if not rid or rid in seen:
            raise ValueError(f"line {line_number}: invalid/duplicate id")
        seen.add(rid)
        file_path=(base/str(row.get("file",""))).resolve()
        if base.resolve() not in file_path.parents or not file_path.is_file():
            raise ValueError(f"{rid}: unsafe/missing image path")
        with Image.open(file_path) as im:
            rgb=im.convert("RGB")
            if min(rgb.size)<16 or max(rgb.size)>4096:
                raise ValueError(f"{rid}: image sides outside 16..4096")
            pixel_sha=hashlib.sha256(rgb.tobytes()+str(rgb.size).encode()).hexdigest()
        captions=row.get("captions")
        if not isinstance(captions,list) or not captions:
            raise ValueError(f"{rid}: captions must be a non-empty list")
        cleaned=[]
        for c in captions:
            text=" ".join(str(c).split())
            if not text or len(text)>256:
                raise ValueError(f"{rid}: invalid caption")
            if text not in cleaned:cleaned.append(text)
        records.append({
            "id":rid,
            "image_id":rid,
            "path":str(file_path),
            "captions":cleaned,
            "split":str(row.get("split","test")).lower(),
            "category":str(row.get("category","unknown")),
            "pixel_sha256":pixel_sha,
        })
    roles={}
    for r in records:roles.setdefault(r["split"],[]).append(r)
    digest_to_role={}
    for role,items in roles.items():
        for r in items:
            prior=digest_to_role.get(r["pixel_sha256"])
            if prior and prior!=role:
                raise ValueError(f"identical decoded image crosses {prior}/{role}")
            digest_to_role[r["pixel_sha256"]]=role
    return roles

if USE_BYOD:
    if not BYOD_ZIP_PATH:
        raise ValueError("Set BYOD_ZIP_PATH for non-interactive BYOD")
    byod_roles=load_byod(BYOD_ZIP_PATH)
    if WORKSHOP_TIER=="FULL" and not {"train","validation","test"}<=set(byod_roles):
        raise ValueError("FULL BYOD requires train/validation/test roles")
    print({k:len(v) for k,v in byod_roles.items()})
else:
    print("BYOD disabled; canonical VizWiz workflow completed.")

## BYOD privacy

User-supplied photographs and captions are processed inside the selected notebook runtime and are not submitted to DIMER workers or APIs.

A hosted notebook remains an external compute environment. Do not upload confidential, personal, restricted, security-sensitive, or proprietary image/text data unless authorized.

VizWiz itself contains photographs of everyday surroundings taken by blind users and can include personal spaces, belongings, labels, documents, and people. The sample corpus should therefore be treated as real-world user-generated imagery, not anonymous synthetic data.

# 27. Interpretation boundaries

### Retrieval is gallery-relative

The same model can score much higher over 70 candidates than over 391.

A Recall@1 number without gallery size is incomplete.

### Nearest neighbor is not truth

A dual encoder always has a nearest vector.

A reranker also assigns every examined pair a score.

Neither provides an inherent “none of these” decision.

### Scores are not interchangeable

- SigLIP retrieval: cosine similarity in its 768-d space.
- BLIP ITC: cosine similarity in its 256-d space.
- BLIP ITM: uncalibrated pairwise classifier output.

`0.7` in one is not probabilistically equivalent to `0.7` in another.

### Rerankers cannot recover omitted candidates

If the correct item is absent from the top-K shortlist, BLIP ITM cannot make it rank first.

Always read candidate oracle@K beside reranked R@1.

### Embedding size is an engineering dimension

A larger representation can improve retrieval but increases:

- storage;
- memory bandwidth;
- ANN index footprint.

### VizWiz is one distribution

Results reflect one pinned sample of photographs and crowd-written captions, not all image search tasks.

### Search bias

Vision-language embeddings inherit semantic associations from large web corpora. Search involving people, occupations, demographic attributes, cultures, or sensitive concepts needs separate fairness and application-specific review.

# 28. Try it yourselfs

### Exercise A — gallery size

Why does the same frozen model's R@1 usually decrease as candidate count rises?

### Exercise B — first-stage oracle

If SigLIP 2 candidate oracle@5 is 0.94, what is the highest possible reranked R@1 at K=5?

### Exercise C — cross encoder

Why not run BLIP ITM against every image-caption pair in a million-image corpus?

### Exercise D — dimensions

Would you always choose a 768-dimensional index over a 256-dimensional one if R@1 is slightly higher?

### Exercise E — no match

What should an application do with a nearest neighbor returned for `a satellite orbiting mars` when no such image exists?

### Exercise F — adaptation

If adapted BLIP improves VizWiz rsum but makes another image population worse, did the adaptation succeed?

That is a deployment-domain and regression-policy question, not a single-metric question.

# 29. Export provenance and report bundle

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Final provenance export
import datetime
import shutil

experiment_manifest={
    "notebook_spec":"2.1",
    "notebook_profile":"MULTI-CAPABILITY",
    "notebook_mode":"WORKSHOP",
    "workshop_revision":"0.1.0-candidate",
    "timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "tier":WORKSHOP_TIER,
    "dataset":dataset_manifest,
    "models":frozen["models"],
    "evaluation":frozen["evaluation"],
    "retrieval_test":retrieval_table.to_dict("records"),
    "reranking_test":rerank_table.to_dict("records"),
    "gallery_size":gallery_table.to_dict("records"),
    "runtime_storage":storage_table.to_dict("records"),
    "blip_adaptation":blip_adapter_report,
    "adapted_blip_test":adapted_blip_metrics,
    "adapted_reranking":adapted_rerank_rows,
    "blip_reload":blip_reload_parity,
    "index_manifest":index_manifest,
    "evidence_scope":"one seeded VizWiz-Captions sample; 391-image / 1,737-caption held-out gallery",
    "release_gates":[
        "fresh T4 STANDARD Run all",
        "fresh T4 FULL Run all",
        "record peak VRAM and exact released-notebook wall times",
    ],
}

(OUTPUT_ROOT/"provenance"/"experiment_manifest.json").write_text(
    json.dumps(experiment_manifest,indent=2,default=str),encoding="utf-8"
)
(OUTPUT_ROOT/"workshop_summary.json").write_text(
    json.dumps({
        "tier":WORKSHOP_TIER,
        "dataset_role_digest":dataset_manifest["role_digest"],
        "retrieval":retrieval_table.to_dict("records"),
        "reranking":rerank_table.to_dict("records"),
        "adapted_blip":adapted_blip_metrics,
    },indent=2,default=str),encoding="utf-8"
)

bundle=shutil.make_archive(
    str(Path(OUTPUT_DIR).resolve())+"_DIMER_Vision_Language_Retrieval_Report",
    "zip",
    root_dir=Path(OUTPUT_DIR).resolve(),
)
print({"bundle":bundle,"sha256":sha256_file(bundle)})

# 30. Troubleshooting

| Symptom | Likely cause | Corrective action |
|---|---|---|
| model digest mismatch | incomplete/drifted Hub file | delete cache and retry; never bypass verification |
| VizWiz shard digest mismatch | incomplete/drifted dataset file | remove cached shard and retry |
| test gallery count not 391 / captions not 1,737 | schema/sample drift | refuse; do not silently change gallery |
| SigLIP OOM | image/text batch too large for runtime | use canonical T4 and reduce only embedding batch size, not gallery |
| BLIP reranking slow | pairwise fused passes | lower K only through a new validation/freeze experiment |
| strong metrics on 70 but weaker on 391 | expected gallery-size effect | report both; do not compare unlike galleries |
| ITM worsens a correct coarse top-1 | reranker error | preserve the observed regression; do not tune on test |
| correct item outside top-K | candidate-generator miss | reranker cannot recover it; read oracle@K |
| adapter digest mismatch | corrupt/foreign adapter | refuse to load |
| BYOD image appears in two roles | leakage | fix split ownership before running |

Explicit failure is preferable to changing gallery membership or metric definitions after seeing results.

## Try it yourself — one controlled change

Use the same experimental discipline as the canonical path:

**Predict → change one variable → rerun → observe → explain**

Use the existing caption-perturbation section. Before running it, predict how removing or changing one visually important phrase should affect the retrieved images. Change only the caption, rerun retrieval/reranking, then explain which ranking changes are attributable to the query and which are limited by the original candidate set.

Keep exploratory changes separate from the frozen canonical test result.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

Retrieval is gallery-relative: the system chooses among available candidates rather than deciding an image is universally correct. A reranker can reorder candidates but cannot recover an image omitted by the first-stage retriever. Similarity and ITM scores should not be read as interchangeable calibrated probabilities.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Compare the principal held-out retrieval result with the analytical/non-neural references, report at least one Recall@K or rank-based measure, describe what reranking changed, and note the gallery-size, category, or hard-negative behavior that most affects interpretation.


# Glossary

| Term | Meaning |
|---|---|
| **Dual encoder** | Independently encodes image and text into a shared vector space |
| **Cross encoder / fused matcher** | Scores an image-text pair jointly with cross-modal interaction |
| **ITC** | Image-text contrastive representation/objective |
| **ITM** | Image-text matching pairwise classifier |
| **I2T** | Image-to-text retrieval |
| **T2I** | Text-to-image retrieval |
| **Recall@K** | Fraction of queries with at least one correct candidate in the first K results |
| **Median rank** | Median position of the first correct result |
| **rsum** | Sum of I2T/T2I R@1, R@5 and R@10 |
| **Candidate oracle@K** | Fraction of queries whose correct item exists somewhere in the top-K shortlist |
| **Hard negative** | High-scoring incorrect candidate |
| **Embedding index** | Stored vectors and their item-order/provenance mapping |
| **Reranker** | More expensive model that reorders a small candidate shortlist |
| **Sample-sanity evidence** | Bounded notebook evidence, not a benchmark/deployment guarantee |

In [ ]:
# @title Run-all completion summary
summary={
    "notebook_spec":"2.1",
    "profile":"MULTI-CAPABILITY",
    "mode":"WORKSHOP",
    "tier":WORKSHOP_TIER,
    "models":["SigLIP v1","SigLIP 2","BLIP ITC/ITM"],
    "test_gallery_images":len(test_records),
    "test_gallery_captions":len(test_texts),
    "rerank_top_k":RERANK_TOP_K,
    "index_directory":str((OUTPUT_ROOT/"index").resolve()),
    "output_directory":str(OUTPUT_ROOT.resolve()),
    "release_status":"candidate",
}
display(pd.Series(summary,name="value").to_frame())
print(
    "Vision-language retrieval workshop complete. "
    "Interpret results as one-gallery sample-sanity evidence, not a universal model ranking."
)

# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |
